RUNTIME: CPU (no GPU needed for this notebook)    
INSTRUCTIONS:

1. Set runtime to CPU: Runtime → Change runtime type → None
2. Run all cells in order: Runtime → Run all
3. When prompted, authorize Google Drive access
4. Model checkpoints are saved to Google Drive via:
```
    import joblib
    joblib.dump(model, SAVE_DIR + "checkpoints/svm_model.joblib")
```
Make sure SAVE_DIR is set correctly in the setup cell    

5. To save results/figures to GitHub:    
    a. Right-click the file in the left sidebar → Download    
    b. Go to the GitHub repo → results/ → Add file → Upload files    
    c. Commit directly on GitHub
6. Save the notebook: Ctrl+S → save to GitHub when prompted

In [1]:
# ── 1. Standard imports ──────────────────────────────────────────
import ast
import random
import sys
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from datasets import load_dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.multiclass import OneVsRestClassifier
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.svm import LinearSVC

# Resolve repo root so src imports work in local Jupyter and Colab
repo_candidates = [Path.cwd(), Path.cwd().parent]
repo_root = None
for candidate in repo_candidates:
    if (candidate / "src" / "evaluate.py").exists():
        repo_root = candidate
        break

if repo_root is None:
    raise FileNotFoundError("Could not find repo root containing src/evaluate.py")

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.evaluate import evaluate_run

print("Repo root:", repo_root)


/home/arand/miniconda3/envs/cs4120/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Repo root: /mnt/c/Users/arand/OneDrive/Desktop/NEU/cs4120/cs4120-project


In [2]:
# ── 2. Data loading helpers ──────────────────────────────────────
def _resolve_data_dir():
    for candidate in [repo_root / "data", Path("../data"), Path("data")]:
        if (candidate / "train.csv").exists() and (candidate / "test.csv").exists():
            return candidate
    raise FileNotFoundError("Could not find data directory with train.csv/test.csv")

def _parse_labels_column(df, label_col="labels"):
    df = df.copy()

    def _parse_one(value):
        if isinstance(value, list):
            return [int(x) for x in value]
        if isinstance(value, np.ndarray):
            return [int(x) for x in value.tolist()]
        if isinstance(value, str):
            s = value.strip()
            if s.startswith('[') and s.endswith(']'):
                body = s[1:-1].strip()
                if body == "":
                    return []
                if ',' in body:
                    toks = [t.strip() for t in body.split(',') if t.strip()]
                else:
                    toks = [t for t in body.split() if t]
                return [int(t) for t in toks]
            return [int(x) for x in ast.literal_eval(s)]
        return value

    df[label_col] = df[label_col].apply(_parse_one)
    return df

def _choose_text_col(df):
    for col in ["text_clean_tfidf", "text"]:
        if col in df.columns:
            return col
    raise ValueError("No text column found. Expected one of: text_clean_tfidf, text")

def _get_label_names(train_df):
    # Prefer canonical label names from HF feature schema.
    try:
        ds = load_dataset("google-research-datasets/go_emotions", "simplified")
        return ds["train"].features["labels"].feature.names
    except Exception:
        max_id = max(max(labels) if labels else -1 for labels in train_df["labels"])
        return [f"label_{i}" for i in range(max_id + 1)]

data_dir = _resolve_data_dir()
results_dir = repo_root / "results"
results_dir.mkdir(parents=True, exist_ok=True)

print("Data dir:", data_dir)
print("Results dir:", results_dir)


Data dir: /mnt/c/Users/arand/OneDrive/Desktop/NEU/cs4120/cs4120-project/data
Results dir: /mnt/c/Users/arand/OneDrive/Desktop/NEU/cs4120/cs4120-project/results


In [3]:
# ── 3. Experiment configuration ──────────────────────────────────
METHOD_NAME = "svm_tfidf"
DATA_FRACTIONS = [0.01, 0.05, 0.10, 0.25, 0.50, 1.00]
SEEDS = [42, 7, 21]

TFIDF_CONFIG = {
    "max_features": 10000,
    "ngram_range": (1, 2),
}
SVM_CONFIG = {
    "class_weight": "balanced",
}

def _fraction_to_pct_tag(frac):
    return f"{int(round(float(frac) * 100))}pct"

def _train_file_for_fraction_seed(frac, seed):
    frac = float(frac)
    if frac == 1.0:
        return data_dir / "train.csv"

    pct_tag = _fraction_to_pct_tag(frac)
    seeded = data_dir / f"train_{pct_tag}_seed{seed}.csv"
    canonical = data_dir / f"train_{pct_tag}.csv"

    if seeded.exists():
        return seeded
    if canonical.exists():
        return canonical

    raise FileNotFoundError(f"No train file found for frac={frac}, seed={seed}")

print("Configured fractions:", DATA_FRACTIONS)
print("Configured seeds:", SEEDS)


Configured fractions: [0.01, 0.05, 0.1, 0.25, 0.5, 1.0]
Configured seeds: [42, 7, 21]


In [4]:
# ── 4. Training + Evaluation loop (shared framework) ────────────
def _set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)

test_df = _parse_labels_column(pd.read_csv(data_dir / "test.csv"), label_col="labels")
text_col = _choose_text_col(test_df)

# Use full train split to infer label space robustly
train_full_df = _parse_labels_column(pd.read_csv(data_dir / "train.csv"), label_col="labels")
label_names = _get_label_names(train_full_df)

mlb = MultiLabelBinarizer(classes=list(range(len(label_names))))
mlb.fit([list(range(len(label_names)))])
y_test = mlb.transform(test_df["labels"])

all_overall = []
all_per_class = []
run_errors = []

checkpoints_dir = results_dir / "checkpoints" / "svm_tfidf"
checkpoints_dir.mkdir(parents=True, exist_ok=True)

for seed in SEEDS:
    for frac in DATA_FRACTIONS:
        print(f"\n=== SVM run: fraction={frac}, seed={seed} ===")
        try:
            _set_seed(seed)
            train_path = _train_file_for_fraction_seed(frac, seed)
            train_df = _parse_labels_column(pd.read_csv(train_path), label_col="labels")

            if text_col not in train_df.columns:
                text_col = _choose_text_col(train_df)

            y_train = mlb.transform(train_df["labels"])

            vectorizer = TfidfVectorizer(
                max_features=TFIDF_CONFIG["max_features"],
                ngram_range=TFIDF_CONFIG["ngram_range"],
            )

            train_text = train_df[text_col].fillna("").astype(str)
            test_text = test_df[text_col].fillna("").astype(str)

            X_train = vectorizer.fit_transform(train_text)
            X_test = vectorizer.transform(test_text)

            svm = OneVsRestClassifier(
                LinearSVC(random_state=int(seed), class_weight=SVM_CONFIG["class_weight"])
            )
            svm.fit(X_train, y_train)

            if float(frac) == 1.0:
                joblib.dump(
                    (vectorizer, svm),
                    checkpoints_dir / f"svm_model_seed{seed}.joblib",
                )

            y_pred = svm.predict(X_test)

            evaluation = evaluate_run(
                method=METHOD_NAME,
                data_fraction=float(frac),
                seed=int(seed),
                label_names=label_names,
                y_true=y_test,
                y_pred=y_pred,
            )

            all_overall.append(evaluation["overall"])
            all_per_class.append(evaluation["per_class"])

            macro_f1 = float(evaluation["overall"].iloc[0]["macro_f1"])
            print(f"Completed: macro_f1={macro_f1:.4f}")

        except Exception as exc:
            run_errors.append({"seed": seed, "data_fraction": frac, "error": str(exc)})
            print(f"FAILED: {exc}")

print("\nTraining loop complete.")


Generating test split: 100%|██████████| 5427/5427 [00:00<00:00, 539459.36 examples/s]



=== SVM run: fraction=0.01, seed=42 ===
Completed: macro_f1=0.1462

=== SVM run: fraction=0.05, seed=42 ===
Completed: macro_f1=0.2790

=== SVM run: fraction=0.1, seed=42 ===
Completed: macro_f1=0.3239

=== SVM run: fraction=0.25, seed=42 ===
Completed: macro_f1=0.3630

=== SVM run: fraction=0.5, seed=42 ===
Completed: macro_f1=0.3459

=== SVM run: fraction=1.0, seed=42 ===
Completed: macro_f1=0.3421

=== SVM run: fraction=0.01, seed=7 ===
Completed: macro_f1=0.1569

=== SVM run: fraction=0.05, seed=7 ===
Completed: macro_f1=0.2733

=== SVM run: fraction=0.1, seed=7 ===


/home/arand/miniconda3/envs/cs4120/lib/python3.13/site-packages/sklearn/svm/_base.py:1258: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/home/arand/miniconda3/envs/cs4120/lib/python3.13/site-packages/sklearn/svm/_base.py:1258: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


Completed: macro_f1=0.3217

=== SVM run: fraction=0.25, seed=7 ===
Completed: macro_f1=0.3506

=== SVM run: fraction=0.5, seed=7 ===
Completed: macro_f1=0.3571

=== SVM run: fraction=1.0, seed=7 ===
Completed: macro_f1=0.3421

=== SVM run: fraction=0.01, seed=21 ===
Completed: macro_f1=0.1427

=== SVM run: fraction=0.05, seed=21 ===
Completed: macro_f1=0.2870

=== SVM run: fraction=0.1, seed=21 ===
Completed: macro_f1=0.3435

=== SVM run: fraction=0.25, seed=21 ===
Completed: macro_f1=0.3634

=== SVM run: fraction=0.5, seed=21 ===
Completed: macro_f1=0.3650

=== SVM run: fraction=1.0, seed=21 ===
Completed: macro_f1=0.3421

Training loop complete.


In [5]:
# ── 5. Save results ──────────────────────────────────────────────
if all_overall:
    overall_df = pd.concat(all_overall, ignore_index=True).sort_values(["seed", "data_fraction"])
    per_class_df = pd.concat(all_per_class, ignore_index=True).sort_values(["seed", "data_fraction", "emotion"])

    overall_path = results_dir / "svm_tfidf_overall.csv"
    per_class_path = results_dir / "svm_tfidf_per_class.csv"
    readme_path = results_dir / "svm_tfidf_results.csv"

    overall_df.to_csv(overall_path, index=False)
    per_class_df.to_csv(per_class_path, index=False)

    # README-compatible long format
    readme_df = per_class_df[["method", "data_fraction", "seed", "emotion", "f1", "precision", "recall"]].copy()
    readme_df.to_csv(readme_path, index=False)

    print("Saved outputs:")
    print(overall_path)
    print(per_class_path)
    print(readme_path)

    display(overall_df.head())
    display(per_class_df.head())
else:
    print("No successful runs; no result files were saved.")

if run_errors:
    print("\nRun errors:")
    display(pd.DataFrame(run_errors))
else:
    print("\nAll runs completed without exceptions.")


Saved outputs:
/mnt/c/Users/arand/OneDrive/Desktop/NEU/cs4120/cs4120-project/results/svm_tfidf_overall.csv
/mnt/c/Users/arand/OneDrive/Desktop/NEU/cs4120/cs4120-project/results/svm_tfidf_per_class.csv
/mnt/c/Users/arand/OneDrive/Desktop/NEU/cs4120/cs4120-project/results/svm_tfidf_results.csv


,method,data_fraction,seed,accuracy,macro_f1,micro_f1,hamming_loss
6,svm_tfidf,0.01,7,0.162705,0.156924,0.269940,0.041683
7,svm_tfidf,0.05,7,0.258154,0.273305,0.408291,0.041143
8,svm_tfidf,0.10,7,0.265156,0.321713,0.425491,0.046632
9,svm_tfidf,0.25,7,0.244150,0.350610,0.436543,0.052561
10,svm_tfidf,0.50,7,0.209324,0.357100,0.427107,0.060695


,method,data_fraction,seed,emotion,precision,recall,f1,support,tp,fp,fn,tn
168,svm_tfidf,0.01,7,admiration,0.568627,0.172619,0.264840,504,87,66,417,4857
169,svm_tfidf,0.01,7,amusement,0.803030,0.401515,0.535354,264,106,26,158,5137
170,svm_tfidf,0.01,7,anger,0.512195,0.106061,0.175732,198,21,20,177,5209
171,svm_tfidf,0.01,7,annoyance,0.162162,0.018750,0.033613,320,6,31,314,5076
172,svm_tfidf,0.01,7,approval,0.271429,0.054131,0.090261,351,19,51,332,5025



All runs completed without exceptions.
